<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Laboratorios/03_Lab_naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Laboratorio: Naive Bayes Bernoulli para detectar spam

## Objetivo
Construir desde cero un clasificador **Naive Bayes Bernoulli** a partir de un corpus pequeño de correos.

Al terminar debes poder explicar y calcular:

$
P(Y),\qquad P(X_i\mid Y),\qquad P(X\mid Y),\qquad P(Y\mid X)
$

y comprender por qué la hipótesis

$
X_i \perp X_j\mid Y
$

reduce drásticamente la complejidad del modelo.

## Contexto

Queremos clasificar un correo como

$
Y=1:\ \text{spam}, \qquad Y=0:\ \text{normal}.
$

Usaremos cinco palabras del vocabulario:


$V=\{\text{dinero, gratis, premio, proyecto, reunion}\}.$

Cada correo se representa por

$X=(X_1,\ldots,X_5),$

donde

$
X_i=\begin{cases}
1 & \text{si aparece la palabra }i,\\
0 & \text{si no aparece.}
\end{cases}
$
Por ejemplo, "dinero gratis" se representa como $(1,1,0,0,0).$


In [21]:
corpus = [
    ("spam",   "gana dinero gratis"),
    ("spam",   "dinero gratis ahora"),
    ("spam",   "premio dinero gratis"),
    ("spam",   "gana premio ahora"),
    ("spam",   "en la reunion habra dinero gratis"),
    ("normal",  "daremos un premio despues de la reunion"),
    ("normal", "reunion de proyecto"),
    ("normal", "proyecto para mañana"),
    ("normal", "reunion mañana"),
    ("normal", "informe del proyecto"),
    ("normal", "informe del proyecto"),
]

vocabulario = ["dinero", "gratis", "premio", "proyecto", "reunion"]

## Parte 1 — Construir el vector \(X\)

1. Escribe una función `vectorizar(texto, vocabulario)` que transforme un correo en un vector binario.
2. Vectoriza todos los correos del corpus.
3. Verifica manualmente al menos dos ejemplos.

Ejemplo esperado:


$\text{"gana dinero gratis"}\longrightarrow(1,1,0,0,0)$


In [22]:
# TODO: implementa vectorizar(texto, vocabulario)
import numpy as np

In [23]:
# función para vectorizar un texto dado nuestro vocabulario 
def vectorizar(texto, vocabulario):
    vector = np.zeros(len(vocabulario)) # inicializamos el vector
    for i, palabra in enumerate(vocabulario): # iteramos sobre el vocabulario
        if palabra in texto: # si la palabra está en el texto, ponemos un 1 en la posición correspondiente
            vector[i] = 1
    return vector

In [24]:
for i in corpus:
    print(i[0], vectorizar(i[1], vocabulario))

spam [1. 1. 0. 0. 0.]
spam [1. 1. 0. 0. 0.]
spam [1. 1. 1. 0. 0.]
spam [0. 0. 1. 0. 0.]
spam [1. 1. 0. 0. 1.]
normal [0. 0. 1. 0. 1.]
normal [0. 0. 0. 1. 1.]
normal [0. 0. 0. 1. 0.]
normal [0. 0. 0. 0. 1.]
normal [0. 0. 0. 1. 0.]
normal [0. 0. 0. 1. 0.]


## Parte 2 — Calcular el prior \(P(Y)\)

Calcula:


$P(Y=\text{spam}),\qquad P(Y=\text{normal}).$


Recuerda:

$
P(Y=y)=\frac{\#\text{correos de clase }y}{\#\text{correos totales}}.
$

In [25]:
# TODO: calcula P(Y=spam) y P(Y=normal)
# p_spam se define como el número de correos spam dividido entre el total de correos en el corpus

p_spam = sum(1 for i in corpus if i[0] == "spam") / len(corpus) # probabilidad de spam
p_normal = sum(1 for i in corpus if i[0] == "normal") / len(corpus) # probabilidad de normal

## Parte 3 — Calcular $P(X_i=1\mid Y)$

Para cada palabra calcula su frecuencia dentro de cada clase. Por ejemplo:

$
P(X_{dinero}=1\mid Y=spam)
=\frac{\#\text{spam que contienen dinero}}{\#\text{spam}}.
$

Construye una tabla con las cinco palabras y ambas clases.

**Pregunta:** ¿qué ocurre si una palabra nunca aparece en una clase?

In [26]:
# TODO: calcula las probabilidades condicionales sin suavizado
def frecuencia_spam(palabra):
    # de forma explicita:
    #n = 0
    #for i in corpus:
    #    if i[0]=='spam' and palabra in i[1]:
    #        n+=1
    #    else:
    #        n+=0
    # de forma compacta:
    # si la clase es spam y la palabra está en el texto, sumamos 1, de lo contrario sumamos 0
    return sum(1 for i in corpus if i[0]=='spam' and palabra in i[1]) / sum(1 for i in corpus if i[0]=='spam')


def frecuencia_normal(palabra):
    return sum(1 for i in corpus if i[0]=='normal' and palabra in i[1]) / sum(1 for i in corpus if i[0]=='normal')
    
# construimos la tabla de probabilidades condicionales
tabla = {}
for palabra in vocabulario:
    tabla[palabra] = {
        "spam": frecuencia_spam(palabra),
        "normal": frecuencia_normal(palabra)
    }


print(tabla)

{'dinero': {'spam': 0.8, 'normal': 0.0}, 'gratis': {'spam': 0.8, 'normal': 0.0}, 'premio': {'spam': 0.4, 'normal': 0.16666666666666666}, 'proyecto': {'spam': 0.0, 'normal': 0.6666666666666666}, 'reunion': {'spam': 0.2, 'normal': 0.5}}


## Parte 4 — Suavizado de Laplace

Para evitar probabilidades exactamente iguales a cero, usa suavizado de Laplace para variables Bernoulli:


$\hat P(X_i=1\mid Y=y)=\frac{N_{iy}+1}{N_y+2},$


donde $(N_{iy})$ es el número de correos de clase \(y\) que contienen la palabra $(i)$, y $(N_y)$ es el número total de correos de esa clase.

Calcula de nuevo la tabla.

In [27]:
# crear funcion de suavizado de laplace
def suavizado(palabra,clase):
    N_iy = sum(1 for i in corpus if i[0] == clase and palabra in i[1])
    N_y = sum(1 for i in corpus if i[0] == clase)
    return (N_iy + 1) / (N_y + 2)


tabla_suavizado = {}
for palabra in vocabulario:
    tabla_suavizado[palabra] = {
        "spam": suavizado(palabra,"spam"),
        "normal": suavizado(palabra,"normal")
    }

print(tabla_suavizado)

{'dinero': {'spam': 0.7142857142857143, 'normal': 0.125}, 'gratis': {'spam': 0.7142857142857143, 'normal': 0.125}, 'premio': {'spam': 0.42857142857142855, 'normal': 0.25}, 'proyecto': {'spam': 0.14285714285714285, 'normal': 0.625}, 'reunion': {'spam': 0.2857142857142857, 'normal': 0.5}}


## Parte 5 — Clasificar un correo nuevo

Clasifica:

> **"dinero gratis"**

Su vector es

$x=(1,1,0,0,0).$

Bajo Naive Bayes Bernoulli:

$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$

Recuerda que para una palabra ausente:

$P(X_i=0\mid Y=y)=1-P(X_i=1\mid Y=y).$


Calcula los scores conjuntos:

$S_y=P(Y=y)P(x\mid Y=y),$

y finalmente:

$P(Y=y\mid x)=\frac{S_y}{S_{spam}+S_{normal}}.$

Decide la clase del correo.

In [28]:
# TODO: como computar likelihood, score conjunto y posterior para un correo
# clasificar 

def clasificar(correo):
    vec=vectorizar(correo,vocabulario) # vectorizamos el correo

    # Naive-Bayes para spam , calculamos la probabilidad de que el correo sea spam dado el vector
    likelihood_spam = 1
    for i, palabra in enumerate(vocabulario):
        if vec[i] == 1:
            likelihood_spam *= suavizado(palabra, "spam")
        else:
            likelihood_spam *= (1 - suavizado(palabra, "spam"))

    # Naive-Bayes para normal, calculamos la probabilidad de que el correo sea normal dado el vector
    likelihood_normal = 1
    for i, palabra in enumerate(vocabulario):
        if vec[i] == 1:
            likelihood_normal *= suavizado(palabra, "normal")
        else:
            likelihood_normal *= (1 - suavizado(palabra, "normal"))

    # score conjunto
    score_spam = likelihood_spam * p_spam
    score_normal = likelihood_normal * p_normal

    # finalmente, calculamos la probabilidad
    Pspam = score_spam / (score_spam + score_normal)
    Pnormal = score_normal / (score_spam + score_normal)

    if Pspam > Pnormal:
        return "spam"
    else:
        return "normal"


In [29]:
clasificar("dinero gratis")

'spam'

## Parte 6 — Interpretación

Responde brevemente:

1. ¿Dónde se usa la hipótesis de independencia condicional?


La proabilidad condiciones se usa en el paso 5 cuando implementamos el algoritmo de clasificacion completo. Asumimos que las palabras $X_i$ son condicionalmente independientes dado Y:

$$P(x\mid Y=y)=\prod_iP(X_i=x_i\mid Y=y).$$


2. ¿Por qué no necesitamos almacenar una probabilidad para cada uno de los \(2^5\) vectores posibles?

La hipótesis de Naive-Beyes permite obtener todos los vectores conociendo solo las probabilidades de cada palabra individual. Con esto, el problema se reduce de tener que almacenar $2^5 = 32$ vectores a solo conocer 5 valores de probabilidad (teniendo en cuenta que $P(X_i = 0 \mid Y) =  1 - P(X_i = 1 \mid Y)$ ).


3. ¿Por qué Naive Bayes se considera un modelo **generativo** aunque aquí lo usemos para clasificar?

El modelo de Naive Bayes se considera generativo pues estámos modelando como se generan los datos dentro de cada clase , es decir, la cantidad $P(x \mid y)$, y la probabilidad a priori de cada clase, $P(y)$.  Estos dos resultados los aplicamos junto con el teorema de Bayes para obtener $P(y \mid x)$. En principo, con este modelo podemos muestrear nuevos datos para cada clase. 

A diferencia de un modelo discriminativo, como por ejemplo la regresión logística, donde se aprende directamente $P(y \mid x)$ sin modelar como se generan los datos $x$. 